# TemporalBench Benchmark — Temporal Reasoning

This notebook runs the **TemporalBench** benchmark to evaluate the system's ability to reason over events in time, testing concepts like recency, state-at-time, and event sequences.

In [ ]:
# --- Environment setup (must run first) ---
from nb_helpers.config import setup_environment, load_config
setup_environment()

from helpers.types import BenchmarkConfig

# --- Configuration ---
METHODS = ["vdb"]
MAX_QUESTIONS = 10
TOP_K = 5
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200
SKIP_INDEX = False

config_dict = load_config(overrides={
    "methods": METHODS,
    "max_questions": MAX_QUESTIONS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
})
config = BenchmarkConfig.from_dict(config_dict)

BENCHMARK = "temporalbench"
RUN_CONFIG = {
    "num_questions": MAX_QUESTIONS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
}
print(f"Config: {len(METHODS)} methods, {MAX_QUESTIONS} questions, top_k={TOP_K}")

In [ ]:
# --- Load Dataset ---
# Placeholder for TemporalBench dataset loading
from helpers.chunking import chunk_corpus

corpus_text = """
[2023-01-01] Alice joined the company.
[2023-06-15] Alice was promoted to Senior Engineer.
[2024-02-10] Alice became Engineering Manager.
"""

chunks = chunk_corpus(corpus_text, CHUNK_SIZE, CHUNK_OVERLAP, source="temporalbench_sample")

testset = [
    {
        "question": "What is Alice's most recent role?",
        "ground_truth": "Engineering Manager"
    },
    {
        "question": "When was Alice promoted to Senior Engineer?",
        "ground_truth": "2023-06-15"
    }
]

print(f"TemporalBench sample: {len(corpus_text):,} chars -> {len(chunks)} chunks")
print(f"Test set: {len(testset)} questions")

In [ ]:
# --- Run Benchmarks ---
from nb_helpers.pipeline import run_all_methods, save_results

all_results = await run_all_methods(
    methods=METHODS,
    chunks=chunks,
    testset=testset,
    config=config,
    skip_index=SKIP_INDEX,
)
save_results(all_results, BENCHMARK, RUN_CONFIG)

In [ ]:
# --- Evaluate Results ---
from nb_helpers.metrics import evaluate_all_methods

all_results = await evaluate_all_methods(
    all_results,
    config=config,
    use_ragas=False,
    use_em_f1=True,
)
save_results(all_results, BENCHMARK, RUN_CONFIG)

In [ ]:
# --- Summary Table ---
from nb_helpers.charts import summary_table

summary_table(all_results)